In [ ]:
import functions
import os
from data import data, path_map, masks, path_masks, color_corrections
import healpy as hp
import numpy as np
from astropy.io import fits

In [ ]:
# Reload functions module to get latest changes
import importlib
importlib.reload(functions)

In [ ]:
# Default configuration
nside = 512
n_sim = 100
path_save = path_map + 'PYSM/'

quijote_bands = ['11', '13', '17', '19']
wmap_bands = ['23', '33', '41', '61', '94']
planck_bands = ['30', '44', '70', '100', '143', '217', '353']


band_list = quijote_bands + wmap_bands + planck_bands

name_suffix = '_full_bin_20-199'

# Differential Assemblies (DAs) per frequency band
BANDS = {
    'K': ['K1'],
    'Ka': ['Ka1'],
    'Q': ['Q1', 'Q2'],
    'V': ['V1', 'V2'],
    'W': ['W1', 'W2', 'W3', 'W4'],
}

lmax = 2 * nside - 1
dl = 10

mask_select = masks['QUIJOTE_galcut']['galcut10']
mask_name = mask_select['name']
use_simulated_maps = False
use_white_noise = False
use_noise = False # Use noise simulations instead of the HMDM
out_path = '/home/pablo/Desktop/master/tfm/spectra/'
path_spectra = os.path.join(out_path, f'power_spectra_{mask_name}{name_suffix}.fits')
if use_noise:
    path_hmdm_spectra = os.path.join(out_path, f'power_spectra_{mask_name}_noise_sim{name_suffix}.fits')
else:
    path_hmdm_spectra = os.path.join(out_path, f'power_spectra_{mask_name}_hmdm{name_suffix}.fits')
path_avg_std_skyplusnoise = os.path.join(out_path, f'spectra_avg_std_{mask_name}_avg_std{n_sim}_skyplusnoise{name_suffix}.fits')
path_avg_std_noise = os.path.join(out_path, f'spectra_avg_std_{mask_name}_avg_std{n_sim}_noise{name_suffix}.fits')
# Store per-simulation spectra compressed on disk (Astropy supports .fits.gz transparently)
path_full_skyplusnoise = os.path.join(out_path, f'spectra_full_{mask_name}_{n_sim}_skyplusnoise{name_suffix}.fits.gz')
path_full_noise = os.path.join(out_path, f'spectra_full_{mask_name}_{n_sim}_noise{name_suffix}.fits.gz')
path_corrected_spectra = os.path.join(out_path, f'corrected_power_spectra_{mask_name}{name_suffix}.fits')

# mask = hp.read_map(mask_select['path'])

#Create binning scheme
ell_1 = [20, 40, 60, 80, 100, 120, 140, 160, 180]
ell_2 = [39, 59, 79, 99, 119, 139, 159, 179, 199]

binning_params = {
    'type': 'edges',  #'linear' or 'edges'
    'lmax': lmax,
    'dl': dl,
    # For edges
    'ell1': ell_1,
    'ell2': ell_2
}

$\textbf{Part 1: Simulated maps using PySM}$

In [ ]:

# Generate maps
functions.generate_sky_maps(nside, path_save, experiment_select='QUIJOTE', band_select=quijote_bands)
functions.generate_sky_maps(nside, path_save, experiment_select='WMAP', band_select=wmap_bands)
functions.generate_sky_maps(nside, path_save, experiment_select='Planck', band_select=planck_bands)

$\textbf{Part 2: Simulated white noise maps}$

In [ ]:
functions.white_noise_maps(data, nside, experiment_select='QUIJOTE', band_select=quijote_bands, n_sim=n_sim, path_map=path_map)
functions.white_noise_maps(data, nside, experiment_select='WMAP', band_select=wmap_bands, n_sim=n_sim, path_map=path_map)
functions.white_noise_maps(data, nside, experiment_select='Planck', band_select=planck_bands, n_sim=n_sim, path_map=path_map)

$\textbf{Part 3: Build HMDM}$

In [ ]:
base_dir = os.path.dirname(data['WMAP']['23']['hmdm'])
save_path = os.path.dirname(data['WMAP']['23']['hmdm'])

combined_1to4 = functions.coadd_year_range(base_dir=base_dir, year_1=1, year_2=4, save=True, save_path=save_path)
combined_5to9 = functions.coadd_year_range(base_dir=base_dir, year_1=5, year_2=9, save=True, save_path=save_path)

In [ ]:
functions.make_hmdm(data, band_list, save=True)

$\textbf{Part 4: Build beams WMAP}$

In [ ]:
beam_path = os.path.dirname(data['WMAP']['23']['beam'])
save_path = os.path.dirname(data['WMAP']['23']['beam'])

functions.generate_band_beams(
    BANDS,
    beam_path=beam_path,
    save_path=save_path,
    data_dict=data,
    use_qu=True,     # use noise_QU for polarization beams
    mask_path=None
)

$\textbf{Part 5: Compute power spactra}$

In [ ]:
# 1. Prepare binning scheme
b = functions.create_binning(binning_params)

# 2. Precompute workspaces
workspaces = functions.prepare_workspaces(mask, b, nside, lmax=lmax, purify_e=True, purify_b=True)

In [ ]:
# 3. Compute all spectra
spectra_matrix = functions.compute_all_power_spectra(
    data, band_list, mask, b,
    use_simulated_maps=use_simulated_maps,
    use_white_noise=use_white_noise,
    noise_realization=1,
    only_noise=False,
    workspaces=workspaces,
    lmax=lmax
)

# 4. Save spectra matrix into a FITS file
functions.save_spectra_to_fits(spectra_matrix, band_list, out_file=path_spectra)

In [ ]:
# 5. Compute HMDM spectra
hmdm_spectra_matrix = functions.compute_hmdm_power_spectra(
    data, band_list, mask, b, workspaces=workspaces, lmax=lmax, use_noise=use_noise
)

# 6. Save spectra matrix into a FITS file
functions.save_spectra_to_fits(hmdm_spectra_matrix, band_list, out_file=path_hmdm_spectra)

In [ ]:
# 7. Read spectra matrix from FITS
spectra_dict = functions.read_spectra_from_fits(path_spectra, band_list)

In [ ]:
# 8. Compute mean and std over noise realizations (sky + noise)
avg_std_skyplusnoise_dict = functions.average_and_std_spectra(
    data, spectra_dict, band_list, mask, b,
    use_white_noise=use_white_noise,
    n_sim=n_sim, 
    only_noise=False,
    workspaces=workspaces,
    lmax=lmax,
    capture_sims=True  # capture per-simulation arrays for full FITS
)


# 9. Compute mean and std for noise-only maps
avg_std_noise_dict = functions.average_and_std_spectra(
    data, spectra_dict, band_list, mask, b,
    use_white_noise=use_white_noise,
    n_sim=n_sim,
    only_noise=True,
    workspaces=workspaces,
    lmax=lmax,
    capture_sims=True  # capture per-simulation arrays for full FITS
)

In [ ]:
functions.save_sims_to_fits(
    avg_std_skyplusnoise_dict,
    band_list,
    out_file=path_full_skyplusnoise,
    use_white_noise=use_white_noise,
    sims_npz=None,
    avg_std_out_file=path_avg_std_skyplusnoise,
    dtype='float32',
    sims_layout='vector',
    hdu_grouping='by_bandpair',
)

functions.save_sims_to_fits(
    avg_std_noise_dict,
    band_list,
    out_file=path_full_noise,
    use_white_noise=use_white_noise,
    sims_npz=None,
    avg_std_out_file=path_avg_std_noise,
    dtype='float32',
    sims_layout='vector',
    hdu_grouping='by_bandpair',
)

$\textbf{Part 6: Correct power spectra}$

In [ ]:
# Path to Planck CMB spectrum (best-fit cosmology)
cmb_spectrum_file = '/home/pablo/Desktop/master/tfm/spectra/COM_PowerSpect_CMB-base-plikHM-TTTEEE-lowl-lowE-lensing-minimum-theory_R3.01.txt'

corr_spectra, out_file = functions.correct_power_spectra(
    path_spectra, path_avg_std_skyplusnoise, path_avg_std_noise,
    band_list, data, nside, 
    correct_beam=True, 
    correct_unit=True,                    # Convert to K²_RJ (Planck 2018 convention)
    correct_pixel=True, 
    save=True, 
    path_out_file=path_corrected_spectra,
    use_white_noise=use_white_noise, 
    path_hmdm_spectra=path_hmdm_spectra,
    subtract_cmb=True,                    # Subtract Planck CMB spectrum
    cmb_spectrum_path=cmb_spectrum_file   # Path to real Planck spectrum
)

$\textbf{Part 8: MCMC fitting}$

In [ ]:
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

In [ ]:
# -------------------------------
# Fitting configuration (power-law with Gaussian priors and cross-only pairs)
# -------------------------------
from functions import set_gaussian_priors

# Gaussian priors:
# - beta_s ~ N(-3.1, 0.18)  synchrotron spectral index (frequency)
# - beta_d ~ N(1.55, 0.05)  dust spectral index (frequency)
# - alpha_s ~ N(-3.0, 0.30) synchrotron ell-slope (same for EE/BB)
# - alpha_d ~ N(-2.48, 0.20) dust ell-slope (average of Planck ~-2.42 EE and ~-2.54 BB)
#   Dust temperature T_d is fixed internally at 19.6 K (effectively a delta prior).

# set_gaussian_priors({
#     'beta_s': (-3.1, 0.18),
#     'beta_d': (1.55, 0.05),
#     'alpha_s': (-3.0, 0.30),
#     'alpha_d': (-2.48, 0.20),
# })

set_gaussian_priors(None)

fitting_mode = 'power-law'

# Multipole range
ell_min = 30
ell_max = 200

# Sampler configuration
nwalkers = 200
ninter = 25000
discard_fraction = 0.5

# Components to fit in the power-law model
fit_components = (
    'sync',
    'dust',
    'cross'
)

# Bands to include in the fit (auto + cross among these)
# QUIJOTE: 11, 13
# WMAP: K(23), Ka(33)
# Planck: LFI 30, HFI 100/143/217/353
quijote_fit_bands = ['11']
wmap_fit_bands = ['23', '33']
planck_fit_bands = ['30', '100', '143', '217', '353']
band_list_fit = quijote_fit_bands + wmap_fit_bands + planck_fit_bands

# # Build cross-only band pairs across all bands (exclude autos)
# band_pairs_cross_all = []
# for i in range(len(band_list)):
#     for j in range(i + 1, len(band_list)):
#         band_pairs_cross_all.append(f"{band_list[i]}_{band_list[j]}")

# # Use cross-only pairs
# band_pairs = band_pairs_cross_all

# Use 'all' to include both autos and crosses within band_list_fit
band_pairs = 'all'

fit_c_terms = False

# Save paths for corner plots
components_str = '_'.join(fit_components)
save_path_EE = f'/home/pablo/Desktop/master/tfm/figures/corner/corner_{mask_name}_{components_str}_EE{name_suffix}_Flavien.pdf'
save_path_BB = f'/home/pablo/Desktop/master/tfm/figures/corner/corner_{mask_name}_{components_str}_BB{name_suffix}_Flavien.pdf'
save_path_EE_BB = f'/home/pablo/Desktop/master/tfm/figures/corner/corner_{mask_name}_{components_str}_EE-BB{name_suffix}_Flavien.pdf'


# Load corrected spectra
spectra_dict = functions.read_corrected_cls(path_corrected_spectra, band_list_fit)

In [ ]:
# Prepare EE data and run MCMC (power-law with prior and cross-only pairs)
fit_data_EE = functions.prepare_mcmc_data(
    spectra_dict,
    band_list=band_list_fit,
    modes=['EE'],
    ell_min=ell_min,
    ell_max=ell_max,
    band_pairs=band_pairs
)

# # Build Covariance
# cov_info_EE = functions.build_block_diagonal_cov_inv(
#     path_sims_fits=path_full_skyplusnoise,
#     fit_data=fit_data_EE,
#     modes=['EE'],
#     quijote_bands_11_13=['11', '13'],
#     quijote_bands_17_19=[],
#     path_noise_sims=path_full_noise 
# )

In [ ]:
sampler_EE, samples_full_EE, samples_free_EE, param_map_EE, chi2_reduced_EE = functions.run_mcmc(
    fit_data=fit_data_EE,
    fit_components=fit_components,
    fit_c_terms=fit_c_terms,
    nwalkers=nwalkers,
    ninter=ninter,
    discard_fraction=discard_fraction,
    verbose=True,
    fit_mode=fitting_mode,
    color_correction=True,
    cov_matrix=None,
)

In [ ]:
# Plot and save the corner plot
fig_EE = functions.plot_corner(samples_free_EE, param_map_EE, save_path_EE, title=f'EE Mode')

In [ ]:
# Prepare BB data and run MCMC (power-law with prior and cross-only pairs)
fit_data_BB = functions.prepare_mcmc_data(
    spectra_dict,
    band_list=band_list_fit,
    modes=['BB'],
    ell_min=ell_min,
    ell_max=ell_max,
    band_pairs=band_pairs
)


In [ ]:
# Run MCMC with the selected fitting mode
sampler_BB, samples_full_BB, samples_free_BB, param_map_BB, chi2_reduced_BB = functions.run_mcmc(
    fit_data=fit_data_BB,
    fit_components=fit_components,
    fit_c_terms=fit_c_terms,
    nwalkers=nwalkers,
    ninter=ninter,
    discard_fraction=discard_fraction,
    verbose=True,  # Show progress bar
    fit_mode=fitting_mode,
    color_correction=True,
    cov_matrix=None,
)

In [ ]:
# Plot and save the corner plot
fig_BB = functions.plot_corner(samples_free_BB, param_map_BB, save_path_BB, title=f'BB Mode')

$\textbf{Create fitting results table}$

In [ ]:
# ========================================
# Create LaTeX table with fitting results
# ========================================

# Complete table with both WMAP+Planck and QUIJOTE+WMAP+Planck results
# Make sure to run the WMAP+Planck fitting cells first!

results_for_table = [
    # WMAP+Planck results
    {
        'data_label': 'WMAP+Planck',
        'mode': 'EE',
        'samples_free': samples_free_EE_wmap,
        'param_map': param_map_EE_wmap
    },
    {
        'data_label': 'WMAP+Planck',
        'mode': 'BB',
        'samples_free': samples_free_BB_wmap,
        'param_map': param_map_BB_wmap
    },
    # QUIJOTE+WMAP+Planck results
    {
        'data_label': 'QUIJOTE+WMAP+Planck',
        'mode': 'EE',
        'samples_free': samples_free_EE,
        'param_map': param_map_EE
    },
    {
        'data_label': 'QUIJOTE+WMAP+Planck',
        'mode': 'BB',
        'samples_free': samples_free_BB,
        'param_map': param_map_BB
    },
]

# Generate the table
table_save_path = f'/home/pablo/Desktop/master/tfm/tables/fitting_results_{mask_name}{name_suffix}.tex'

latex_table = functions.create_fitting_results_table(
    results_for_table,
    save_path=table_save_path,
    caption=None,  # Use default caption
    label='tab:fit_galcut10',
    ell_range='30--200',  # Update based on your actual ell_min and ell_max
    mask_name=r'$10^{\circ}$ Galactic cut'
)

print("LaTeX table:")
print(latex_table)

In [ ]:
# Optional: Print a summary of the parameter constraints for quick viewing
print("\n" + "="*80)
print("SUMMARY OF FITTING RESULTS")
print("="*80)

for result in results_for_table:
    print(f"\n{result['data_label']} - {result['mode']} mode:")
    print("-" * 60)
    
    samples = result['samples_free']
    param_map = result['param_map']
    
    # Handle both param_map formats
    if len(param_map) > 0 and isinstance(param_map[0], tuple):
        param_names = [name for name, is_free in param_map if is_free]
    else:
        param_names = list(param_map)
    
    for i, pname in enumerate(param_names):
        if i >= samples.shape[1]:
            continue
        median = np.median(samples[:, i])
        lower = np.percentile(samples[:, i], 16)
        upper = np.percentile(samples[:, i], 84)
        
        # Apply scaling for display
        if pname == 'A_s':
            median *= 1e6
            lower *= 1e6
            upper *= 1e6
            unit = 'μK²'
        elif pname == 'A_d':
            median *= 1e9
            lower *= 1e9
            upper *= 1e9
            unit = '10⁻³ μK²'
        else:
            unit = ''
        
        print(f"  {pname:10s}: {median:.3f} +{upper-median:.3f} -{median-lower:.3f} {unit}")

print("\n" + "="*80)

**Instructions for WMAP+Planck fitting:**

To complete the table, you need to:
1. Run the same fitting procedure (EE and BB modes) using only WMAP and Planck bands (without QUIJOTE)
2. Store the results in variables like `samples_free_EE_wmap`, `param_map_EE_wmap`, etc.
3. Uncomment the WMAP+Planck entries in the cell above and rerun the table generation

The WMAP+Planck fit configuration should use:
```python
# Only WMAP and Planck bands (no QUIJOTE)
wmap_planck_bands = wmap_fit_bands + planck_fit_bands
# Then use this in prepare_mcmc_data with band_list=wmap_planck_bands
```

$\textbf{WMAP+Planck only fitting (for comparison table)}$

In [ ]:
# ========================================
# WMAP+Planck fitting (without QUIJOTE)
# ========================================

# Band configuration for WMAP+Planck only
wmap_planck_bands = wmap_fit_bands + planck_fit_bands

# Prepare EE data for WMAP+Planck
fit_data_EE_wmap = functions.prepare_mcmc_data(
    spectra_dict,
    band_list=wmap_planck_bands,
    modes=['EE'],
    ell_min=ell_min,
    ell_max=ell_max,
    band_pairs='all'
)

# Run MCMC for EE mode (WMAP+Planck)
sampler_EE_wmap, samples_full_EE_wmap, samples_free_EE_wmap, param_map_EE_wmap, chi2_reduced_EE_wmap = functions.run_mcmc(
    fit_data=fit_data_EE_wmap,
    fit_components=fit_components,
    fit_c_terms=fit_c_terms,
    nwalkers=nwalkers,
    ninter=ninter,
    discard_fraction=discard_fraction,
    verbose=True,
    fit_mode=fitting_mode,
    color_correction=True,
    cov_matrix=None,
)

In [ ]:
# Plot corner for EE mode (WMAP+Planck)
save_path_EE_wmap = f'/home/pablo/Desktop/master/tfm/figures/corner/corner_{mask_name}_{components_str}_EE{name_suffix}_WMAP_Planck.pdf'
fig_EE_wmap = functions.plot_corner(samples_free_EE_wmap, param_map_EE_wmap, save_path_EE_wmap, title='EE Mode (WMAP+Planck)')

In [ ]:
# Prepare BB data for WMAP+Planck
fit_data_BB_wmap = functions.prepare_mcmc_data(
    spectra_dict,
    band_list=wmap_planck_bands,
    modes=['BB'],
    ell_min=ell_min,
    ell_max=ell_max,
    band_pairs='all'
)

# Run MCMC for BB mode (WMAP+Planck)
sampler_BB_wmap, samples_full_BB_wmap, samples_free_BB_wmap, param_map_BB_wmap, chi2_reduced_BB_wmap = functions.run_mcmc(
    fit_data=fit_data_BB_wmap,
    fit_components=fit_components,
    fit_c_terms=fit_c_terms,
    nwalkers=nwalkers,
    ninter=ninter,
    discard_fraction=discard_fraction,
    verbose=True,
    fit_mode=fitting_mode,
    color_correction=True,
    cov_matrix=None,
)

In [ ]:
# Plot corner for BB mode (WMAP+Planck)
save_path_BB_wmap = f'/home/pablo/Desktop/master/tfm/figures/corner/corner_{mask_name}_{components_str}_BB{name_suffix}_WMAP_Planck.pdf'
fig_BB_wmap = functions.plot_corner(samples_free_BB_wmap, param_map_BB_wmap, save_path_BB_wmap, title='BB Mode (WMAP+Planck)')

$\textbf{Joint EE-BB fitting}$

In [ ]:
# Prepare EE-BB data and run MCMC
fit_data_EE_BB = functions.prepare_mcmc_data(
    spectra_dict,
    band_list=band_list_fit,
    modes=['EE', 'BB'],
    ell_min=ell_min,
    ell_max=ell_max,
    band_pairs=band_pairs
)

# Run MCMC with the selected fitting mode
sampler_EE_BB, samples_full_EE_BB, samples_free_EE_BB, param_map_EE_BB, chi2_reduced_EE_BB = functions.run_mcmc(
    fit_data=fit_data_EE_BB,
    fit_components=fit_components,
    fit_c_terms=fit_c_terms,
    nwalkers=nwalkers,
    ninter=ninter,
    discard_fraction=discard_fraction,
    verbose=True,  # Show progress bar
    fit_mode=fitting_mode,
    color_correction=True,
    joint_analysis = True
)

In [ ]:
# Plot and save the corner plot
fig_joint = functions.plot_corner(
    samples_free_EE_BB, param_map_EE_BB,save_path_EE_BB, title='Joint EE-BB Analysis'
)

$\textbf{Part 9: Bin to bin fitting}$

In [ ]:
# Configuration for bin-to-bin fitting (Gaussian priors + cross-only pairs)
from functions import set_gaussian_priors

# Apply the same Gaussian priors:
# - beta_s ~ N(-3.1, 0.18)
# - beta_d ~ N(1.55, 0.05)
# - alpha_s ~ N(-3.0, 0.30)
# - alpha_d ~ N(-2.48, 0.20)  (average of Planck alpha_EE ~ -2.42 and alpha_BB ~ -2.54)
#   T_d fixed at 19.6 K inside the dust scaling (implicit delta prior)
set_gaussian_priors({
    'beta_s': (-3.1, 0.30),
    # 'beta_d': (1.55, 0.05),
    # 'alpha_s': (-3.0, 0.30),
    # 'alpha_d': (-2.48, 0.20),
})

fit_mode_btb = 'bin-to-bin'  # Use 'bin-to-bin' mode

# Multipole range
ell_min_btb = 20
ell_max_btb = 200

# Sampler configuration
nwalkers_btb = 100
ninter_btb = 15000
discard_fraction_btb = 0.5

# Fit synchrotron, dust, and cross components per bin
fit_components_btb = ('sync', 'dust', 'cross')

# # Build cross-only band pairs across all bands (exclude autos)
# band_pairs_cross_all = []
# for i in range(len(band_list)):
#     for j in range(i + 1, len(band_list)):
#         band_pairs_cross_all.append(f"{band_list[i]}_{band_list[j]}")

# band_pairs_btb = band_pairs_cross_all

band_pairs_btb = 'all'

# Load corrected spectra
spectra_dict_btb = functions.read_corrected_cls(path_corrected_spectra, band_list_fit)

In [ ]:
# Prepare EE data and run bin-to-bin MCMC (cross-only pairs)
fit_data_EE = functions.prepare_mcmc_data(
    spectra_dict_btb,
    band_list=band_list_fit,
    modes=['EE'],
    ell_min=ell_min_btb,
    ell_max=ell_max_btb,
    band_pairs=band_pairs_btb
)

# Run bin-to-bin MCMC for EE mode
samplers_EE, samples_full_EE, samples_free_EE, param_names_btb, chi2_reduced_EE = functions.run_mcmc(
    fit_data=fit_data_EE,
    fit_components=fit_components_btb,
    fit_c_terms=False,
    nwalkers=nwalkers_btb,
    ninter=ninter_btb,
    discard_fraction=discard_fraction_btb,
    verbose=True,
    fit_mode=fit_mode_btb,
    color_correction=True
)

In [ ]:
# Prepare BB data and run bin-to-bin MCMC (cross-only pairs)
fit_data_BB = functions.prepare_mcmc_data(
    spectra_dict_btb,
    band_list=band_list_fit,
    modes=['BB'],
    ell_min=ell_min_btb,
    ell_max=ell_max_btb,
    band_pairs=band_pairs_btb
)

# Run bin-to-bin MCMC for BB mode
samplers_BB, samples_full_BB, samples_free_BB, param_names_btb, chi2_reduced_BB = functions.run_mcmc(
    fit_data=fit_data_BB,
    fit_components=fit_components_btb,
    fit_c_terms=False,
    nwalkers=nwalkers_btb,
    ninter=ninter_btb,
    discard_fraction=discard_fraction_btb,
    verbose=True,
    fit_mode=fit_mode_btb,
    color_correction=True
)

In [ ]:
table_save_path_latex = f'/home/pablo/Desktop/master/tfm/tables/bin_to_bin_results_{mask_name}{name_suffix}.tex'

# Generate LaTeX table
table_latex = functions.create_bin_to_bin_table(
    fit_data_EE=fit_data_EE,
    fit_data_BB=fit_data_BB,
    samples_free_list_EE=samples_free_EE,
    samples_free_list_BB=samples_free_BB,
    param_names=param_names_btb,
    ell1=ell_1,
    ell2=ell_2,
    save_path=table_save_path_latex,
    format='latex'
)

# Generate ASCII table for quick viewing
table_ascii = functions.create_bin_to_bin_table(
    fit_data_EE=fit_data_EE,
    fit_data_BB=fit_data_BB,
    samples_free_list_EE=samples_free_EE,
    samples_free_list_BB=samples_free_BB,
    param_names=param_names_btb,
    ell1=ell_1,
    ell2=ell_2,
    save_path=None,
    format='ascii'
)

print(table_ascii)

In [ ]:
# Plot parameter evolution with ell
plot_save_path = f'/home/pablo/Desktop/master/tfm/figures/bin-to-bin/bin_to_bin_evolution_{mask_name}{name_suffix}.pdf'

fig = functions.plot_bin_to_bin_results(
    fit_data_EE=fit_data_EE,
    fit_data_BB=fit_data_BB,
    samples_free_list_EE=samples_free_EE,
    samples_free_list_BB=samples_free_BB,
    param_names=param_names_btb,
    chi2_reduced_EE=chi2_reduced_EE,
    chi2_reduced_BB=chi2_reduced_BB,
    save_path=plot_save_path,
    figsize=(14, 10)
)

$\textbf{Convergence check for bin-to-bin}$

In [ ]:
convergence_save_path = f'/home/pablo/Desktop/master/tfm/figures/bin-to-bin/bin_to_bin_convergence_{mask_name}{name_suffix}.pdf'

fig_convergence = functions.plot_bin_to_bin_convergence(
    samplers_EE=samplers_EE,
    samplers_BB=samplers_BB,
    ell_1=ell_1,
    ell_2=ell_2,
    ninter=ninter_btb,
    discard_fraction=discard_fraction_btb,
    save_path=convergence_save_path,
    figsize=(14, 10)
)

In [ ]:
path_theoretical_spectra = os.path.join(out_path, f'corrected_theoretical_power_spectra_{mask_name}.fits')

spectra_dict = functions.read_corrected_cls(path_corrected_spectra, band_list)
# theoretical_spectra_dict = functions.read_spectra_from_fits(path_theoretical_spectra, band_list)

ell = spectra_dict['11_11']['ell_eff'][3:20]
cl_11_11   = spectra_dict['11_11']['EE']['SPECTRUM'][3:20]
cl_353_353 = spectra_dict['353_353']['EE']['SPECTRUM'][3:20]
cl_11_353  = spectra_dict['11_353']['EE']['SPECTRUM'][3:20]

# cl_11_11_theo   = theoretical_spectra_dict['11_11']['EE'][3:20]
# cl_353_353_theo = theoretical_spectra_dict['353_353']['EE'][3:20]
# cl_11_353_theo  = theoretical_spectra_dict['11_353']['EE'][3:20]


rho = cl_11_353 / np.sqrt(cl_11_11 * cl_353_353)
# rho_theo = cl_11_353_theo / np.sqrt(cl_11_11_theo * cl_353_353_theo)
print(rho)
# print(cl_11_11)
# print(cl_353_353)
# print(cl_11_353)
# print(rho_theo)